# Install Dependencies

Clean installation that avoids touching Colab's pre-installed packages, pin the requests version to suppress the most common one. The -q flag also suppresses the verbose conflict output, easier to spot real errors.

In [12]:
!pip install -q "langchain==0.2.16" langchain-openai langchain-chroma langchain-community \
             langchain-core langchain-text-splitters \
             chromadb openai tiktoken PyPDF2 gradio scikit-learn \
             "requests==2.32.4"

# Imports & API Keys

In [13]:
import os
import numpy as np
import PyPDF2
from PyPDF2 import PdfReader
import tiktoken
import gradio as gr

from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.chains import ConversationalRetrievalChain

from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Upload and Read a PDF File

The PDF-file is stored only in the Colab virtual machine — which means it will be lost when the session ends or disconnects. If you want the PDF to persist, mount Google Drive and read from there instead:

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# reader = PdfReader("/content/drive/MyDrive/your-folder/designscience.pdf")
# text = ""
# for page in reader.pages:
#     text += page.extract_text()
# print(f"Extracted {len(text):,} characters from PDF.")

In [14]:
from google.colab import files

print("Upload your PDF file:")
uploaded = files.upload()  # Opens a file picker dialog

# Get the filename of the uploaded file
pdf_filename = list(uploaded.keys())[0]

reader = PdfReader(pdf_filename)
text = ""
for page in reader.pages:
    text += page.extract_text()

print(f"Extracted {len(text):,} characters from PDF.")

Upload your PDF file:


Saving designscience.pdf to designscience (1).pdf
Extracted 480,895 characters from PDF.


# Chunk the Text

In [15]:
chunk_size = 8000
chunk_overlap = 200

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

chunks = splitter.split_text(text)
print(f"Number of chunks: {len(chunks)}")
print(chunks[0])

Number of chunks: 62
Paul/uni00A0Johannesson/uni00A0· Erik/uni00A0Perjons
An Introduction to 
Design 
ScienceAn Introduction to Design ScienceThiS is a FM Blank PagePaul Johannesson Erik Perjons
An Introduction to
Design SciencePaul Johannesson
Erik Perjons
Stockholm UniversityKista
Sweden
ISBN 978-3-319-10631-1 ISBN 978-3-319-10632-8 (eBook)
DOI 10.1007/978-3-319-10632-8Springer Cham Heidelberg New York Dordrecht London
Library of Congress Control Number: 2014951854
©Springer International Publishing Switzerland 2014
This work is subject to copyright. All rights are reserved by the Publisher, whether the whole or part
of the material is concerned, speciﬁcally the rights of translation, reprinting, reuse of illustrations,
recitation, broadcasting, reproduction on microﬁlms or in any other physical way, and transmission orinformation storage and retrieval, electronic adaptation, computer software, or by similar or dissimilarmethodology now known or hereafter developed. Exempted from th

# Token Count

In [16]:
encoding = tiktoken.encoding_for_model("gpt-4o")
total_tokens = sum(len(encoding.encode(chunk)) for chunk in chunks)
print(f"Total tokens across all chunks: {total_tokens:,}")

Total tokens across all chunks: 103,984


# Build Vector Store

In [17]:
MODEL = "gpt-4.1"
db_name = "vector_db"

documents = [Document(page_content=chunk) for chunk in chunks]
embeddings = OpenAIEmbeddings()

# Clear existing collection if present
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=db_name
)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

# Check embedding dimensions
sample_embedding = vectorstore._collection.get(limit=1, include=["embeddings"])["embeddings"][0]
print(f"Embedding dimensions: {len(sample_embedding):,}")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Vectorstore created with 62 documents
Embedding dimensions: 1,536


# Set up RAG conversation chain

In [18]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

llm = ChatOpenAI(temperature=0.7, model_name=MODEL)
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
retriever = vectorstore.as_retriever()

conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

# Launch Gradio chat UI

In [19]:
def chat(message, history):
    result = conversation_chain.invoke({"question": message})
    return result["answer"]

# share=True is required in Colab — it creates a public tunnel URL
gr.ChatInterface(chat, type="messages").launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0832de264adb947025.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
